# Global Mobility Application Analyzer
## US Visa Approval Prediction - Colab Edition

**Pipeline:** Data Ingestion -> Data Validation -> Data Transformation -> Model Trainer -> Prediction -> Export

> AWS / S3 / Docker / MongoDB removed. All data is local. Artifacts saved to Colab filesystem.

In [1]:
# Install required packages
# evidently removed - drift detection uses scipy.stats.ks_2samp instead
# (evidently 0.2.8 is incompatible with NumPy 2.x which Colab ships with)
!pip install -q pandas numpy scikit-learn imbalanced-learn dill pyyaml from-root mlflow dagshub
print("All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Set environment variables from Colab Secrets
# Add your secrets via: Colab left sidebar -> Key icon -> 'Add new secret'
# Keys needed: MONGO_DB_URL, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME, MLFLOW_TRACKING_PASSWORD
import os
from google.colab import userdata

# MongoDB connection URL (used if you re-enable MongoDB ingestion)
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# MLflow / DagsHub tracking
USE_DAGSHUB = True  # Set to False to log locally inside Colab instead

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow: logs saved to /content/mlruns
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('Env vars set.')
print(f"  MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

Env vars set.
  MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


In [4]:
# Upload the dataset CSV (Visadataset.csv or visa.csv)
from google.colab import files
import os, shutil
print("Please upload your Visadataset.csv file")
uploaded = files.upload()
uploaded_name = list(uploaded.keys())[0]

target_file_name = "Visadataset.csv"
current_file_path = os.path.join(os.getcwd(), uploaded_name)
target_file_path = os.path.join(os.getcwd(), target_file_name)

# Check if the uploaded file is already the target file and in the correct place
# os.path.samefile checks if two paths point to the same inode (same file)
if os.path.exists(target_file_path) and os.path.samefile(current_file_path, target_file_path):
    print(f"File '{uploaded_name}' is already at '{target_file_path}'. Skipping rename/move.")
else:
    # If the target_file_path already exists (e.g., from a previous run with a different uploaded_name), remove it first
    if os.path.exists(target_file_path):
        os.remove(target_file_path)
    shutil.move(current_file_path, target_file_path)
    print(f"Moved '{uploaded_name}' to '{target_file_path}'.")

print(f"Saved as {target_file_path} ({os.path.getsize(target_file_path):,} bytes)")

Please upload your Visadataset.csv file


Saving Visadataset.csv to Visadataset (1).csv
Moved 'Visadataset (1).csv' to '/content/Visadataset.csv'.
Saved as /content/Visadataset.csv (1,855,358 bytes)


In [5]:
# Create project directory structure (mirrors original repo minus cloud components)
import os
dirs = ['visa/components','visa/entity','visa/pipeline','visa/utils',
        'visa/exception','visa/logger','visa/constants','visa/data_access',
        'visa/configuration','config','logs','artifact','static/css']
for d in dirs:
    os.makedirs(f"/content/{d}", exist_ok=True)
packages = ['visa','visa/components','visa/entity','visa/pipeline','visa/utils',
            'visa/exception','visa/logger','visa/constants','visa/data_access','visa/configuration']
for pkg in packages:
    init = f"/content/{pkg}/__init__.py"
    if not os.path.exists(init): open(init,'w').close()
print("Project structure created!")

Project structure created!


In [6]:
# Write config/schema.yaml - defines column types, encoders, drop columns
schema = (
    'columns:\n'
    '  - case_id: category\n'
    '  - continent: category\n'
    '  - education_of_employee: category\n'
    '  - has_job_experience: category\n'
    '  - requires_job_training: category\n'
    '  - no_of_employees: int\n'
    '  - yr_of_estab: int\n'
    '  - region_of_employment: category\n'
    '  - prevailing_wage: int\n'
    '  - unit_of_wage: category\n'
    '  - full_time_position: category\n'
    '  - case_status: category\n\n'
    'numerical_columns:\n'
    '  - no_of_employees\n'
    '  - prevailing_wage\n'
    '  - yr_of_estab\n\n'
    'categorical_columns:\n'
    '  - case_id\n'
    '  - continent\n'
    '  - education_of_employee\n'
    '  - has_job_experience\n'
    '  - requires_job_training\n'
    '  - region_of_employment\n'
    '  - unit_of_wage\n'
    '  - full_time_position\n'
    '  - case_status\n\n'
    'drop_columns:\n'
    '  - case_id\n'
    '  - yr_of_estab\n\n'
    'num_features:\n'
    '  - no_of_employees\n'
    '  - prevailing_wage\n'
    '  - company_age\n\n'
    'or_columns:\n'
    '  - has_job_experience\n'
    '  - requires_job_training\n'
    '  - full_time_position\n'
    '  - education_of_employee\n\n'
    'oh_columns:\n'
    '  - continent\n'
    '  - unit_of_wage\n'
    '  - region_of_employment\n\n'
    'transform_columns:\n'
    '  - no_of_employees\n'
    '  - company_age\n'
)
with open('/content/config/schema.yaml','w') as f: f.write(schema)
print('config/schema.yaml written')

config/schema.yaml written


In [7]:
# Write config/model.yaml - models and hyperparameter grids for GridSearchCV
model_yaml = (
    'grid_search:\n'
    '  class: GridSearchCV\n'
    '  module: sklearn.model_selection\n'
    '  params:\n'
    '    cv: 3\n'
    '    verbose: 1\n\n'
    'model_selection:\n'
    '  module_0:\n'
    '    class: KNeighborsClassifier\n'
    '    module: sklearn.neighbors\n'
    '    params:\n'
    '      algorithm: kd_tree\n'
    '      weights: uniform\n'
    '      n_neighbors: 3\n'
    '    search_param_grid:\n'
    '      algorithm:\n'
    '        - auto\n'
    '        - kd_tree\n'
    '      weights:\n'
    '        - uniform\n'
    '        - distance\n'
    '      n_neighbors:\n'
    '        - 3\n'
    '        - 5\n'
    '        - 9\n\n'
    '  module_1:\n'
    '    class: RandomForestClassifier\n'
    '    module: sklearn.ensemble\n'
    '    params:\n'
    '      max_depth: 10\n'
    '      max_features: sqrt\n'
    '      n_estimators: 3\n'
    '    search_param_grid:\n'
    '      max_depth:\n'
    '        - 10\n'
    '        - 15\n'
    '        - 20\n'
    '      max_features:\n'
    '        - sqrt\n'
    '        - log2\n'
    '      n_estimators:\n'
    '        - 3\n'
    '        - 5\n'
    '        - 9\n'
)
with open('/content/config/model.yaml','w') as f: f.write(model_yaml)
print('config/model.yaml written')

config/model.yaml written


In [8]:
# Write all Python module source files to disk
# Each module is stored as a multi-line string and written to its path
import os

def write_module(path, content):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)
    print(f'Written: {path}')

# ---- exception module ----
write_module('/content/visa/exception/__init__.py', '''import sys

def error_message_detail(error, error_detail):
    _, _, exc_tb = error_detail.exc_info()
    fn = exc_tb.tb_frame.f_code.co_filename
    return f"Error in [{fn}] line [{exc_tb.tb_lineno}]: [{error}]"

class USvisaException(Exception):
    def __init__(self, error_message, error_detail):
        super().__init__(error_message)
        self.error_message = error_message_detail(error_message, error_detail)
    def __str__(self):
        return self.error_message
''')

# ---- logger module ----
write_module('/content/visa/logger/__init__.py', '''import logging, os
from datetime import datetime
LOG_DIR = "/content/logs"
os.makedirs(LOG_DIR, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(LOG_DIR, f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"),
    format="[ %(asctime)s ] %(name)s - %(levelname)s - %(message)s",
    level=logging.DEBUG,
)
''')

print('Core modules written!')

Written: /content/visa/exception/__init__.py
Written: /content/visa/logger/__init__.py
Core modules written!


In [9]:
# Write visa/constants/__init__.py
# All project-wide constants. AWS/S3/MongoDB constants removed.
constants_code = '''import os
from datetime import date
DATABASE_NAME    = "VISA_APPLICATION_DATA"
PIPELINE_NAME    = "visa"
ARTIFACT_DIR     = "/content/artifact"
MODEL_FILE_NAME  = "model.pkl"
FILE_NAME        = "visa.csv"
TRAIN_FILE_NAME  = "train.csv"
TEST_FILE_NAME   = "test.csv"
TARGET_COLUMN    = "case_status"
CURRENT_YEAR     = date.today().year
PREPROCSSING_OBJECT_FILE_NAME = "preprocessing.pkl"
SCHEMA_FILE_PATH = os.path.join("/content/config", "schema.yaml")
DATA_INGESTION_COLLECTION_NAME = "visa_data"
DATA_INGESTION_DIR_NAME        = "data_ingestion"
DATA_INGESTION_FEATURE_STORE_DIR = "feature_store"
DATA_INGESTION_INGESTED_DIR    = "ingested"
DATA_INGESTION_TRAIN_TEST_SPLIT_RATIO = 0.2
DATA_VALIDATION_DIR_NAME       = "data_validation"
DATA_VALIDATION_DRIFT_REPORT_DIR = "drift_report"
DATA_VALIDATION_DRIFT_REPORT_FILE_NAME = "report.yaml"
DATA_TRANSFORMATION_DIR_NAME   = "data_transformation"
DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR   = "transformed"
DATA_TRANSFORMATION_TRANSFORMED_OBJECT_DIR = "transformed_object"
MODEL_TRAINER_DIR_NAME          = "model_trainer"
MODEL_TRAINER_TRAINED_MODEL_DIR = "trained_model"
MODEL_TRAINER_TRAINED_MODEL_NAME = "model.pkl"
MODEL_TRAINER_EXPECTED_SCORE     = 0.6
MODEL_TRAINER_MODEL_CONFIG_FILE_PATH = os.path.join("/content/config", "model.yaml")
APP_HOST = "0.0.0.0"
APP_PORT = 8080
'''
with open('/content/visa/constants/__init__.py','w') as f: f.write(constants_code)
print('constants written')

constants written


In [10]:
# Write entity modules: artifact_entity, config_entity, estimator
import os

# --- artifact_entity.py ---
artifact_code = '''from dataclasses import dataclass

@dataclass
class DataIngestionArtifact:
    trained_file_path: str
    test_file_path: str

@dataclass
class DataValidationArtifact:
    validation_status: bool
    message: str
    drift_report_file_path: str

@dataclass
class DataTransformationArtifact:
    transformed_object_file_path: str
    transformed_train_file_path: str
    transformed_test_file_path: str

@dataclass
class ClassificationMetricArtifact:
    f1_score: float
    precision_score: float
    recall_score: float

@dataclass
class ModelTrainerArtifact:
    trained_model_file_path: str
    metric_artifact: ClassificationMetricArtifact
'''
with open('/content/visa/entity/artifact_entity.py','w') as f: f.write(artifact_code)

# --- config_entity.py ---
config_code = '''import os
from dataclasses import dataclass
from datetime import datetime
from visa.constants import *
TIMESTAMP = datetime.now().strftime("%m_%d_%Y_%H_%M_%S")

@dataclass
class TrainingPipelineConfig:
    pipeline_name: str = PIPELINE_NAME
    artifact_dir: str = os.path.join(ARTIFACT_DIR, TIMESTAMP)
    timestamp: str = TIMESTAMP

training_pipeline_config = TrainingPipelineConfig()

@dataclass
class DataIngestionConfig:
    data_ingestion_dir: str = os.path.join(training_pipeline_config.artifact_dir, DATA_INGESTION_DIR_NAME)
    feature_store_file_path: str = os.path.join(data_ingestion_dir, DATA_INGESTION_FEATURE_STORE_DIR, FILE_NAME)
    training_file_path: str = os.path.join(data_ingestion_dir, DATA_INGESTION_INGESTED_DIR, TRAIN_FILE_NAME)
    testing_file_path: str = os.path.join(data_ingestion_dir, DATA_INGESTION_INGESTED_DIR, TEST_FILE_NAME)
    train_test_split_ratio: float = DATA_INGESTION_TRAIN_TEST_SPLIT_RATIO
    collection_name: str = DATA_INGESTION_COLLECTION_NAME

@dataclass
class DataValidationConfig:
    data_validation_dir: str = os.path.join(training_pipeline_config.artifact_dir, DATA_VALIDATION_DIR_NAME)
    drift_report_file_path: str = os.path.join(data_validation_dir, DATA_VALIDATION_DRIFT_REPORT_DIR, DATA_VALIDATION_DRIFT_REPORT_FILE_NAME)

@dataclass
class DataTransformationConfig:
    data_transformation_dir: str = os.path.join(training_pipeline_config.artifact_dir, DATA_TRANSFORMATION_DIR_NAME)
    transformed_train_file_path: str = os.path.join(data_transformation_dir, DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR, TRAIN_FILE_NAME.replace('csv','npy'))
    transformed_test_file_path: str = os.path.join(data_transformation_dir, DATA_TRANSFORMATION_TRANSFORMED_DATA_DIR, TEST_FILE_NAME.replace('csv','npy'))
    transformed_object_file_path: str = os.path.join(data_transformation_dir, DATA_TRANSFORMATION_TRANSFORMED_OBJECT_DIR, PREPROCSSING_OBJECT_FILE_NAME)

@dataclass
class ModelTrainerConfig:
    model_trainer_dir: str = os.path.join(training_pipeline_config.artifact_dir, MODEL_TRAINER_DIR_NAME)
    trained_model_file_path: str = os.path.join(model_trainer_dir, MODEL_TRAINER_TRAINED_MODEL_DIR, MODEL_FILE_NAME)
    expected_accuracy: float = MODEL_TRAINER_EXPECTED_SCORE
    model_config_file_path: str = MODEL_TRAINER_MODEL_CONFIG_FILE_PATH

@dataclass
class USvisaPredictorConfig:
    model_file_path: str = ""
'''
with open('/content/visa/entity/config_entity.py','w') as f: f.write(config_code)

# --- estimator.py ---
estimator_code = '''import sys
from pandas import DataFrame
from visa.exception import USvisaException
from visa.logger import logging

class TargetValueMapping:
    def __init__(self):
        self.Certified = 0
        self.Denied = 1
    def _asdict(self):
        return self.__dict__
    def reverse_mapping(self):
        m = self._asdict()
        return dict(zip(m.values(), m.keys()))

class VisaModel:
    def __init__(self, preprocessing_object, trained_model_object):
        self.preprocessing_object = preprocessing_object
        self.trained_model_object = trained_model_object
    def predict(self, dataframe):
        try:
            tf = self.preprocessing_object.transform(dataframe)
            return self.trained_model_object.predict(tf)
        except Exception as e:
            raise USvisaException(e, sys) from e
    def __repr__(self): return f"{type(self.trained_model_object).__name__}()"
    def __str__(self): return f"{type(self.trained_model_object).__name__}()"
'''
with open('/content/visa/entity/estimator.py','w') as f: f.write(estimator_code)
print('Entity modules written!')

Entity modules written!


In [11]:
# Write visa/utils/main_utils.py - shared helpers: YAML I/O, dill serialization, numpy I/O
utils_code = '''import os, sys
import numpy as np
import dill, yaml
from pandas import DataFrame
from visa.exception import USvisaException
from visa.logger import logging

def read_yaml_file(file_path):
    try:
        with open(file_path, 'rb') as f: return yaml.safe_load(f)
    except Exception as e: raise USvisaException(e, sys) from e

def write_yaml_file(file_path, content, replace=False):
    try:
        if replace and os.path.exists(file_path): os.remove(file_path)
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'w') as f: yaml.dump(content, f)
    except Exception as e: raise USvisaException(e, sys) from e

def load_object(file_path):
    try:
        with open(file_path, 'rb') as f: return dill.load(f)
    except Exception as e: raise USvisaException(e, sys) from e

def save_object(file_path, obj):
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'wb') as f: dill.dump(obj, f)
    except Exception as e: raise USvisaException(e, sys) from e

def save_numpy_array_data(file_path, array):
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'wb') as f: np.save(f, array)
    except Exception as e: raise USvisaException(e, sys) from e

def load_numpy_array_data(file_path):
    try:
        with open(file_path, 'rb') as f: return np.load(f)
    except Exception as e: raise USvisaException(e, sys) from e

def drop_columns(df, cols):
    try: return df.drop(columns=cols, axis=1)
    except Exception as e: raise USvisaException(e, sys) from e
'''
with open('/content/visa/utils/main_utils.py','w') as f: f.write(utils_code)
print('Utils module written!')

Utils module written!


In [12]:
# Write visa/components/data_ingestion.py
# CHANGE vs original: reads from local CSV instead of MongoDB
di_code = '''import os, sys
import pandas as pd
from sklearn.model_selection import train_test_split
from visa.entity.config_entity import DataIngestionConfig
from visa.entity.artifact_entity import DataIngestionArtifact
from visa.exception import USvisaException
from visa.logger import logging

LOCAL_CSV_PATH = "/content/Visadataset.csv"  # uploaded CSV path in Colab

class DataIngestion:
    def __init__(self, data_ingestion_config=DataIngestionConfig()):
        try: self.data_ingestion_config = data_ingestion_config
        except Exception as e: raise USvisaException(e, sys)

    def export_data_into_feature_store(self):
        """Load CSV from LOCAL_CSV_PATH, save to feature store, return DataFrame."""
        try:
            logging.info("Loading data from local CSV (MongoDB not used in Colab)")
            df = pd.read_csv(LOCAL_CSV_PATH)
            fp = self.data_ingestion_config.feature_store_file_path
            os.makedirs(os.path.dirname(fp), exist_ok=True)
            df.to_csv(fp, index=False, header=True)
            return df
        except Exception as e: raise USvisaException(e, sys)

    def split_data_as_train_test(self, dataframe):
        """Split DataFrame into train/test CSVs."""
        try:
            train_set, test_set = train_test_split(
                dataframe, test_size=self.data_ingestion_config.train_test_split_ratio, random_state=42)
            os.makedirs(os.path.dirname(self.data_ingestion_config.training_file_path), exist_ok=True)
            train_set.to_csv(self.data_ingestion_config.training_file_path, index=False, header=True)
            test_set.to_csv(self.data_ingestion_config.testing_file_path, index=False, header=True)
        except Exception as e: raise USvisaException(e, sys) from e

    def initiate_data_ingestion(self):
        """Orchestrate export + split and return DataIngestionArtifact."""
        try:
            df = self.export_data_into_feature_store()
            self.split_data_as_train_test(df)
            return DataIngestionArtifact(
                trained_file_path=self.data_ingestion_config.training_file_path,
                test_file_path=self.data_ingestion_config.testing_file_path)
        except Exception as e: raise USvisaException(e, sys) from e
'''
with open('/content/visa/components/data_ingestion.py','w') as f: f.write(di_code)
print('data_ingestion.py written!')

data_ingestion.py written!


In [13]:
# Write visa/components/data_validation.py
# NOTE: evidently replaced with scipy KS-test drift detection
# Reason: evidently==0.2.8 uses np.float_ which was removed in NumPy 2.0
#         and Colab ships with NumPy 2.x which cannot be downgraded easily.
dv_code = '''import sys
import pandas as pd
import numpy as np
from scipy import stats
from visa.exception import USvisaException
from visa.logger import logging
from visa.utils.main_utils import write_yaml_file, read_yaml_file
from visa.entity.artifact_entity import DataIngestionArtifact, DataValidationArtifact
from visa.entity.config_entity import DataValidationConfig
from visa.constants import SCHEMA_FILE_PATH

class DataValidation:
    """Validates ingested data against schema and checks for statistical drift
    using the Kolmogorov-Smirnov test (replaces Evidently for NumPy 2.x compatibility)."""

    def __init__(self, data_ingestion_artifact, data_validation_config):
        try:
            self.data_ingestion_artifact = data_ingestion_artifact
            self.data_validation_config  = data_validation_config
            self._schema_config = read_yaml_file(SCHEMA_FILE_PATH)
        except Exception as e: raise USvisaException(e, sys)

    def validate_number_of_columns(self, df):
        """Return True if dataframe has the exact number of columns in schema."""
        return len(df.columns) == len(self._schema_config['columns'])

    def is_column_exist(self, df):
        """Return True if all expected numerical and categorical columns are present."""
        missing_num = [c for c in self._schema_config['numerical_columns'] if c not in df.columns]
        missing_cat = [c for c in self._schema_config['categorical_columns'] if c not in df.columns]
        if missing_num: logging.info(f'Missing numerical: {missing_num}')
        if missing_cat: logging.info(f'Missing categorical: {missing_cat}')
        return len(missing_num) == 0 and len(missing_cat) == 0

    @staticmethod
    def read_data(fp): return pd.read_csv(fp)

    def detect_dataset_drift(self, reference_df, current_df, threshold=0.05):
        """Detect drift on numerical columns using Kolmogorov-Smirnov test.
        A column is drifted if KS p-value < threshold.
        Overall dataset drift = True if more than half the columns drift.
        Saves a YAML drift report. Returns True if drift detected."""
        try:
            num_cols = self._schema_config['numerical_columns']
            drift_results = {}
            n_drifted = 0
            for col in num_cols:
                if col in reference_df.columns and col in current_df.columns:
                    stat, p_value = stats.ks_2samp(
                        reference_df[col].dropna().values,
                        current_df[col].dropna().values
                    )
                    is_drifted = bool(p_value < threshold)
                    drift_results[col] = {'ks_statistic': float(stat), 'p_value': float(p_value), 'drifted': is_drifted}
                    if is_drifted: n_drifted += 1
                    logging.info(f'Column {col}: KS={stat:.4f}, p={p_value:.4f}, drifted={is_drifted}')
            overall_drift = n_drifted > len(num_cols) / 2
            report = {
                'drift_detection_method': 'Kolmogorov-Smirnov',
                'threshold': threshold,
                'n_features': len(num_cols),
                'n_drifted_features': n_drifted,
                'dataset_drift': overall_drift,
                'columns': drift_results,
            }
            write_yaml_file(self.data_validation_config.drift_report_file_path, report)
            logging.info(f'Drift: {n_drifted}/{len(num_cols)} columns. Overall: {overall_drift}')
            return overall_drift
        except Exception as e: raise USvisaException(e, sys) from e

    def initiate_data_validation(self):
        """Run all checks and return DataValidationArtifact."""
        try:
            err_msg = ''
            train_df = self.read_data(self.data_ingestion_artifact.trained_file_path)
            test_df  = self.read_data(self.data_ingestion_artifact.test_file_path)
            if not self.validate_number_of_columns(train_df): err_msg += 'Missing cols in train. '
            if not self.validate_number_of_columns(test_df):  err_msg += 'Missing cols in test. '
            if not self.is_column_exist(train_df): err_msg += 'Required cols missing in train. '
            if not self.is_column_exist(test_df):  err_msg += 'Required cols missing in test. '
            status = len(err_msg) == 0
            if status:
                drift = self.detect_dataset_drift(train_df, test_df)
                err_msg = 'Drift detected' if drift else 'Drift not detected'
            return DataValidationArtifact(
                validation_status=status, message=err_msg,
                drift_report_file_path=self.data_validation_config.drift_report_file_path)
        except Exception as e: raise USvisaException(e, sys) from e
'''
with open('/content/visa/components/data_validation.py','w') as f: f.write(dv_code)
print('data_validation.py written (KS-test drift, no evidently)!')

data_validation.py written (KS-test drift, no evidently)!


In [14]:
# Write visa/components/data_transformation.py
# Feature engineering, encoding, scaling, and SMOTEENN resampling
dt_code = '''import sys
import numpy as np, pandas as pd
from imblearn.combine import SMOTEENN
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from visa.constants import TARGET_COLUMN, SCHEMA_FILE_PATH, CURRENT_YEAR
from visa.entity.config_entity import DataTransformationConfig
from visa.entity.artifact_entity import DataTransformationArtifact, DataIngestionArtifact, DataValidationArtifact
from visa.exception import USvisaException
from visa.logger import logging
from visa.utils.main_utils import save_object, save_numpy_array_data, read_yaml_file, drop_columns
from visa.entity.estimator import TargetValueMapping

class DataTransformation:
    def __init__(self, data_ingestion_artifact, data_transformation_config, data_validation_artifact):
        try:
            self.data_ingestion_artifact    = data_ingestion_artifact
            self.data_transformation_config = data_transformation_config
            self.data_validation_artifact   = data_validation_artifact
            self._schema_config = read_yaml_file(SCHEMA_FILE_PATH)
        except Exception as e: raise USvisaException(e, sys)

    @staticmethod
    def read_data(fp): return pd.read_csv(fp)

    def get_data_transformer_object(self):
        """Build ColumnTransformer with OHE, OrdinalEncoder, PowerTransformer, StandardScaler."""
        try:
            oh_cols  = self._schema_config['oh_columns']
            or_cols  = self._schema_config['or_columns']
            tr_cols  = self._schema_config['transform_columns']
            num_cols = self._schema_config['num_features']
            tr_pipe  = Pipeline(steps=[('transformer', PowerTransformer(method='yeo-johnson'))])
            return ColumnTransformer(transformers=[
                ('OneHotEncoder',   OneHotEncoder(),  oh_cols),
                ('Ordinal_Encoder', OrdinalEncoder(), or_cols),
                ('Transformer',     tr_pipe,          tr_cols),
                ('StandardScaler',  StandardScaler(), num_cols),
            ])
        except Exception as e: raise USvisaException(e, sys) from e

    def initiate_data_transformation(self):
        """Engineer features, encode, scale, apply SMOTEENN, save arrays and preprocessor."""
        try:
            if not self.data_validation_artifact.validation_status:
                raise Exception(self.data_validation_artifact.message)
            preprocessor = self.get_data_transformer_object()
            train_df = self.read_data(self.data_ingestion_artifact.trained_file_path)
            test_df  = self.read_data(self.data_ingestion_artifact.test_file_path)
            drop_cols = self._schema_config['drop_columns']
            X_train = train_df.drop(columns=[TARGET_COLUMN], axis=1)
            y_train = train_df[TARGET_COLUMN]
            X_train['company_age'] = CURRENT_YEAR - X_train['yr_of_estab']  # feature engineering
            X_train = drop_columns(df=X_train, cols=drop_cols)
            y_train = y_train.replace(TargetValueMapping()._asdict())
            X_test  = test_df.drop(columns=[TARGET_COLUMN], axis=1)
            y_test  = test_df[TARGET_COLUMN]
            X_test['company_age'] = CURRENT_YEAR - X_test['yr_of_estab']
            X_test  = drop_columns(df=X_test, cols=drop_cols)
            y_test  = y_test.replace(TargetValueMapping()._asdict())
            X_train_arr = preprocessor.fit_transform(X_train)  # fit on train
            X_test_arr  = preprocessor.transform(X_test)       # only transform test
            smt = SMOTEENN(sampling_strategy='minority')        # handle class imbalance
            X_tr_f, y_tr_f = smt.fit_resample(X_train_arr, y_train)
            X_te_f, y_te_f = smt.fit_resample(X_test_arr,  y_test)
            train_arr = np.c_[X_tr_f, np.array(y_tr_f)]
            test_arr  = np.c_[X_te_f, np.array(y_te_f)]
            save_object(self.data_transformation_config.transformed_object_file_path, preprocessor)
            save_numpy_array_data(self.data_transformation_config.transformed_train_file_path, train_arr)
            save_numpy_array_data(self.data_transformation_config.transformed_test_file_path,  test_arr)
            return DataTransformationArtifact(
                transformed_object_file_path=self.data_transformation_config.transformed_object_file_path,
                transformed_train_file_path=self.data_transformation_config.transformed_train_file_path,
                transformed_test_file_path=self.data_transformation_config.transformed_test_file_path)
        except Exception as e: raise USvisaException(e, sys) from e
'''
with open('/content/visa/components/data_transformation.py','w') as f: f.write(dt_code)
print('data_transformation.py written!')

data_transformation.py written!


In [15]:
# Write visa/components/model_trainer.py
# neuro_mf replaced with local ModelFactory using GridSearchCV
# MLflow tracking added: logs params, metrics, and model artifact per run
mt_code = '''import sys, importlib, os
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from visa.exception import USvisaException
from visa.logger import logging
from visa.utils.main_utils import load_numpy_array_data, read_yaml_file, load_object, save_object
from visa.entity.config_entity import ModelTrainerConfig
from visa.entity.artifact_entity import DataTransformationArtifact, ModelTrainerArtifact, ClassificationMetricArtifact
from visa.entity.estimator import VisaModel

class BestModelDetail:
    def __init__(self, best_model, best_score, best_params, model_name):
        self.best_model=best_model; self.best_score=best_score
        self.best_params=best_params; self.model_name=model_name

class ModelFactory:
    """Local replacement for neuro_mf. Reads model.yaml and runs GridSearchCV per model."""
    def __init__(self, model_config_path): self.config = read_yaml_file(model_config_path)
    def _get_estimator(self, module, cls, params):
        return getattr(__import__(module, fromlist=[cls]), cls)(**params)
    def get_best_model(self, X, y, base_accuracy=0.6):
        best_score, best_detail = 0, None
        gs_params = self.config['grid_search'].get('params', {'cv':3,'verbose':1})
        for key, m in self.config['model_selection'].items():
            logging.info(f'Training: {m["class"]}')
            try:
                est = self._get_estimator(m['module'], m['class'], m.get('params',{}))
                gs = GridSearchCV(est, param_grid=m.get('search_param_grid',{}), scoring='f1', **gs_params)
                gs.fit(X, y)
                logging.info(f'  Best CV F1={gs.best_score_:.4f}')
                if gs.best_score_ > best_score:
                    best_score = gs.best_score_
                    best_detail = BestModelDetail(gs.best_estimator_, gs.best_score_, gs.best_params_, m['class'])
            except Exception as e: logging.warning(f'  Skipping {m["class"]}: {e}')
        if not best_detail or best_detail.best_score < base_accuracy:
            raise Exception(f'No model exceeded base accuracy {base_accuracy}. Best: {best_score:.4f}')
        return best_detail

class ModelTrainer:
    """Loads arrays, runs ModelFactory, wraps best model with preprocessor, saves VisaModel.
    Logs all params, metrics, and the model artifact to MLflow."""
    def __init__(self, data_transformation_artifact, model_trainer_config):
        self.data_transformation_artifact = data_transformation_artifact
        self.model_trainer_config = model_trainer_config

    def get_model_object_and_report(self, train, test):
        try:
            mf = ModelFactory(self.model_trainer_config.model_config_file_path)
            X_tr, y_tr = train[:,:-1], train[:,-1]
            X_te, y_te = test[:,:-1],  test[:,-1]
            best = mf.get_best_model(X_tr, y_tr, self.model_trainer_config.expected_accuracy)
            y_pred = best.best_model.predict(X_te)
            metrics = ClassificationMetricArtifact(
                f1_score=f1_score(y_te,y_pred),
                precision_score=precision_score(y_te,y_pred),
                recall_score=recall_score(y_te,y_pred))
            logging.info(f'F1={metrics.f1_score:.4f} | Acc={accuracy_score(y_te,y_pred):.4f}')
            return best, metrics
        except Exception as e: raise USvisaException(e, sys) from e

    def initiate_model_trainer(self):
        try:
            tr = load_numpy_array_data(self.data_transformation_artifact.transformed_train_file_path)
            te = load_numpy_array_data(self.data_transformation_artifact.transformed_test_file_path)
            best, metrics = self.get_model_object_and_report(tr, te)
            pre = load_object(self.data_transformation_artifact.transformed_object_file_path)
            visa_model = VisaModel(preprocessing_object=pre, trained_model_object=best.best_model)
            save_object(self.model_trainer_config.trained_model_file_path, visa_model)

            # ── MLflow: log everything for this training run ──────────────
            mlflow.set_experiment('US_Visa_Approval_Prediction')
            with mlflow.start_run(run_name=best.model_name):
                # Log best hyperparameters found by GridSearchCV
                mlflow.log_params(best.best_params)
                # Log model class name as a tag
                mlflow.set_tag('model_name', best.model_name)
                mlflow.set_tag('best_cv_f1', round(best.best_score, 4))
                # Log test-set metrics
                mlflow.log_metric('f1_score',        metrics.f1_score)
                mlflow.log_metric('precision_score', metrics.precision_score)
                mlflow.log_metric('recall_score',    metrics.recall_score)
                # Log the sklearn estimator (not the VisaModel wrapper)
                mlflow.sklearn.log_model(best.best_model, artifact_path='model')
                # Log the saved model.pkl as an artifact too
                mlflow.log_artifact(self.model_trainer_config.trained_model_file_path, artifact_path='visa_model_pkl')
                # Log the preprocessing pickle
                mlflow.log_artifact(self.data_transformation_artifact.transformed_object_file_path, artifact_path='preprocessing')
                # Log the drift report YAML if it exists
                drift_path = os.path.join(os.path.dirname(self.data_transformation_artifact.transformed_object_file_path)
                    .replace('data_transformation', 'data_validation'), 'drift_report', 'report.yaml')
                if os.path.exists(drift_path):
                    mlflow.log_artifact(drift_path, artifact_path='drift_report')
                logging.info(f'MLflow run logged: {best.model_name}')
            # ─────────────────────────────────────────────────────────────

            return ModelTrainerArtifact(
                trained_model_file_path=self.model_trainer_config.trained_model_file_path,
                metric_artifact=metrics)
        except Exception as e: raise USvisaException(e, sys) from e
'''
with open('/content/visa/components/model_trainer.py','w') as f: f.write(mt_code)
print('model_trainer.py written (with MLflow tracking)!')

model_trainer.py written (with MLflow tracking)!


In [16]:
# Write training and prediction pipeline files
import os

tp_code = '''import sys
from visa.exception import USvisaException
from visa.logger import logging
from visa.components.data_ingestion import DataIngestion
from visa.components.data_validation import DataValidation
from visa.components.data_transformation import DataTransformation
from visa.components.model_trainer import ModelTrainer
from visa.entity.config_entity import DataIngestionConfig, DataValidationConfig, DataTransformationConfig, ModelTrainerConfig

class TrainingPipeline:
    """Orchestrates: DataIngestion -> DataValidation -> DataTransformation -> ModelTrainer
    AWS/S3 stages (ModelEvaluation, ModelPusher) excluded for local Colab execution."""
    def __init__(self):
        self.data_ingestion_config      = DataIngestionConfig()
        self.data_validation_config     = DataValidationConfig()
        self.data_transformation_config = DataTransformationConfig()
        self.model_trainer_config       = ModelTrainerConfig()

    def start_data_ingestion(self):
        try: return DataIngestion(self.data_ingestion_config).initiate_data_ingestion()
        except Exception as e: raise USvisaException(e, sys) from e

    def start_data_validation(self, di):
        try: return DataValidation(di, self.data_validation_config).initiate_data_validation()
        except Exception as e: raise USvisaException(e, sys) from e

    def start_data_transformation(self, di, dv):
        try: return DataTransformation(di, self.data_transformation_config, dv).initiate_data_transformation()
        except Exception as e: raise USvisaException(e, sys)

    def start_model_trainer(self, dt):
        try: return ModelTrainer(dt, self.model_trainer_config).initiate_model_trainer()
        except Exception as e: raise USvisaException(e, sys)

    def run_pipeline(self):
        """Run the complete local pipeline and return ModelTrainerArtifact."""
        try:
            di = self.start_data_ingestion()
            dv = self.start_data_validation(di)
            dt = self.start_data_transformation(di, dv)
            mt = self.start_model_trainer(dt)
            logging.info('Training Pipeline finished successfully')
            return mt
        except Exception as e: raise USvisaException(e, sys)
'''
with open('/content/visa/pipeline/training_pipeline.py','w') as f: f.write(tp_code)

pp_code = '''import sys
import pandas as pd
from visa.exception import USvisaException
from visa.logger import logging
from visa.utils.main_utils import load_object
from visa.entity.estimator import TargetValueMapping

class VisaData:
    """Collects 10 input features and converts to DataFrame for prediction."""
    def __init__(self, continent, education_of_employee, has_job_experience,
                 requires_job_training, no_of_employees, region_of_employment,
                 prevailing_wage, unit_of_wage, full_time_position, company_age):
        self.continent=continent; self.education_of_employee=education_of_employee
        self.has_job_experience=has_job_experience; self.requires_job_training=requires_job_training
        self.no_of_employees=no_of_employees; self.region_of_employment=region_of_employment
        self.prevailing_wage=prevailing_wage; self.unit_of_wage=unit_of_wage
        self.full_time_position=full_time_position; self.company_age=company_age

    def get_usvisa_input_data_frame(self):
        return pd.DataFrame({'continent':[self.continent],'education_of_employee':[self.education_of_employee],
            'has_job_experience':[self.has_job_experience],'requires_job_training':[self.requires_job_training],
            'no_of_employees':[self.no_of_employees],'region_of_employment':[self.region_of_employment],
            'prevailing_wage':[self.prevailing_wage],'unit_of_wage':[self.unit_of_wage],
            'full_time_position':[self.full_time_position],'company_age':[self.company_age]})

class VisaClassifier:
    """Loads saved VisaModel from local path and runs predictions. (AWS S3 removed)"""
    def __init__(self, model_path):
        try:
            self.model = load_object(file_path=model_path)
            logging.info(f'Model loaded: {model_path}')
        except Exception as e: raise USvisaException(e, sys)
    def predict(self, dataframe):
        try: return self.model.predict(dataframe)
        except Exception as e: raise USvisaException(e, sys)
'''
with open('/content/visa/pipeline/prediction_pipeline.py','w') as f: f.write(pp_code)
print('Pipeline modules written!')

Pipeline modules written!


## Running the Pipeline

Each cell runs one pipeline stage. Artifacts are saved under `/content/artifact/<timestamp>/`.

In [17]:
# PIPELINE STAGE 1: Data Ingestion
# Load Visadataset.csv -> save to feature store -> split into train/test CSVs
import sys
sys.path.insert(0, '/content')  # ensure visa package is importable

from visa.components.data_ingestion import DataIngestion
from visa.entity.config_entity import DataIngestionConfig
import pandas as pd

print('Stage 1: Data Ingestion...')
data_ingestion_config   = DataIngestionConfig()
data_ingestion          = DataIngestion(data_ingestion_config)
data_ingestion_artifact = data_ingestion.initiate_data_ingestion()

train_df = pd.read_csv(data_ingestion_artifact.trained_file_path)
test_df  = pd.read_csv(data_ingestion_artifact.test_file_path)
print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(train_df['case_status'].value_counts())
print('Stage 1 complete!')

Stage 1: Data Ingestion...
Train: (20384, 12) | Test: (5096, 12)
case_status
Certified    13617
Denied        6767
Name: count, dtype: int64
Stage 1 complete!


In [18]:
# PIPELINE STAGE 2: Data Validation
# Validates column schema then runs Evidently drift detection between train and test
from visa.components.data_validation import DataValidation
from visa.entity.config_entity import DataValidationConfig

print('Stage 2: Data Validation...')
data_validation_config   = DataValidationConfig()
data_validation          = DataValidation(data_ingestion_artifact, data_validation_config)
data_validation_artifact = data_validation.initiate_data_validation()

print(f'Validation status : {data_validation_artifact.validation_status}')
print(f'Message           : {data_validation_artifact.message}')
print(f'Drift report      : {data_validation_artifact.drift_report_file_path}')
print('Stage 2 complete!')

Stage 2: Data Validation...
Validation status : True
Message           : Drift not detected
Drift report      : /content/artifact/03_18_2026_04_22_27/data_validation/drift_report/report.yaml
Stage 2 complete!


In [19]:
# PIPELINE STAGE 3: Data Transformation
# - Engineers company_age feature from yr_of_estab
# - Applies OneHotEncoder, OrdinalEncoder, PowerTransformer (Yeo-Johnson), StandardScaler
# - Applies SMOTEENN to address class imbalance
# - Saves train.npy, test.npy, preprocessing.pkl
from visa.components.data_transformation import DataTransformation
from visa.entity.config_entity import DataTransformationConfig
import numpy as np

print('Stage 3: Data Transformation (may take ~1 minute)...')
data_transformation_config   = DataTransformationConfig()
data_transformation          = DataTransformation(
    data_ingestion_artifact, data_transformation_config, data_validation_artifact)
data_transformation_artifact = data_transformation.initiate_data_transformation()

tr = np.load(data_transformation_artifact.transformed_train_file_path)
te = np.load(data_transformation_artifact.transformed_test_file_path)
print(f'Train array: {tr.shape} | Test array: {te.shape}')
print('Stage 3 complete!')

Stage 3: Data Transformation (may take ~1 minute)...


/content/visa/components/data_transformation.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace(TargetValueMapping()._asdict())
/content/visa/components/data_transformation.py:61: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test  = y_test.replace(TargetValueMapping()._asdict())


Train array: (13577, 25) | Test array: (3540, 25)
Stage 3 complete!


In [20]:
# PIPELINE STAGE 4: Model Trainer
# GridSearchCV over KNeighborsClassifier and RandomForestClassifier
# Picks model with best cross-validated F1 score
# NOTE: This is the slowest step (~5-15 minutes on Colab CPU)
from visa.components.model_trainer import ModelTrainer
from visa.entity.config_entity import ModelTrainerConfig

print('Stage 4: Model Trainer (GridSearchCV running - please wait)...')
model_trainer_config   = ModelTrainerConfig()
model_trainer          = ModelTrainer(data_transformation_artifact, model_trainer_config)
model_trainer_artifact = model_trainer.initiate_model_trainer()

m = model_trainer_artifact.metric_artifact
print(f'Model saved: {model_trainer_artifact.trained_model_file_path}')
print(f'F1={m.f1_score:.4f} | Precision={m.precision_score:.4f} | Recall={m.recall_score:.4f}')
print('Stage 4 complete!')

Stage 4: Model Trainer (GridSearchCV running - please wait)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 18 candidates, totalling 54 fits


2026/03/18 04:22:58 INFO mlflow.tracking.fluent: Experiment with name 'US_Visa_Approval_Prediction' does not exist. Creating a new experiment.
2026/03/18 04:23:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/18 04:23:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNeighborsClassifier at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/1/runs/de36379f299c4bb58040c66a388cf472
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/1
Model saved: /content/artifact/03_18_2026_04_22_27/model_trainer/trained_model/model.pkl
F1=0.8164 | Precision=0.8356 | Recall=0.7981
Stage 4 complete!


## Sample Predictions

Run inference on custom inputs using the trained model.

In [21]:
# PREDICTION: Run inference on sample applicants
# Edit sample_inputs to test different scenarios
from visa.pipeline.prediction_pipeline import VisaData, VisaClassifier
from visa.entity.estimator import TargetValueMapping

classifier  = VisaClassifier(model_path=model_trainer_artifact.trained_model_file_path)
rev_map     = TargetValueMapping().reverse_mapping()  # {0: 'Certified', 1: 'Denied'}

# Applicants to predict
# Fields: continent, education_of_employee, has_job_experience, requires_job_training,
#         no_of_employees, region_of_employment, prevailing_wage, unit_of_wage,
#         full_time_position, company_age
sample_inputs = [
    dict(continent='Asia', education_of_employee="Master's", has_job_experience='Y',
         requires_job_training='N', no_of_employees=5000, region_of_employment='Northeast',
         prevailing_wage=80000, unit_of_wage='Year', full_time_position='Y', company_age=20),
    dict(continent='Europe', education_of_employee='High School', has_job_experience='N',
         requires_job_training='Y', no_of_employees=150, region_of_employment='South',
         prevailing_wage=12, unit_of_wage='Hour', full_time_position='N', company_age=3),
]

print('=' * 50)
for i, inp in enumerate(sample_inputs, 1):
    df    = VisaData(**inp).get_usvisa_input_data_frame()
    pred  = int(classifier.predict(df)[0])
    label = rev_map.get(pred, 'Unknown')
    icon  = '[APPROVED]' if label == 'Certified' else '[DENIED]'
    print(f'Applicant #{i}: {inp["education_of_employee"]} | exp={inp["has_job_experience"]} | wage={inp["prevailing_wage"]} {inp["unit_of_wage"]} => {icon} {label}')
print('=' * 50)

Applicant #1: Master's | exp=Y | wage=80000 Year => [APPROVED] Certified
Applicant #2: High School | exp=N | wage=12 Hour => [APPROVED] Certified


## Export Artifacts

Zip all outputs and download to your local machine.

In [22]:
# ZIP all artifacts, logs, config, and MLflow runs then download
import shutil, os, zipfile

zip_base = '/content/Global_Mobility_Analyzer_outputs'
print('Creating zip archive...')

# Start with artifact directory
shutil.make_archive(base_name=zip_base, format='zip', root_dir='/content', base_dir='artifact')

# Add logs, config, and mlruns to the same zip
with zipfile.ZipFile(zip_base + '.zip', 'a') as zf:
    for folder in ['logs', 'config', 'mlruns']:
        folder_path = f'/content/{folder}'
        if not os.path.exists(folder_path):
            print(f'  Skipping {folder}/ (not found - MLflow may not have run yet)')
            continue
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, '/content'))
        print(f'  Added {folder}/')

size = os.path.getsize(zip_base + '.zip') / 1024 / 1024
print(f'Archive ready: {zip_base}.zip ({size:.2f} MB)')

# Show what's inside the zip
with zipfile.ZipFile(zip_base + '.zip') as zf:
    top_level = sorted(set(p.split('/')[0] for p in zf.namelist()))
    print(f'Top-level folders in zip: {top_level}')

# Trigger download
from google.colab import files
files.download(zip_base + '.zip')
print('Download triggered!')

Creating zip archive...
  Added logs/
  Added config/
  Skipping mlruns/ (not found - MLflow may not have run yet)
Archive ready: /content/Global_Mobility_Analyzer_outputs.zip (1.95 MB)
Top-level folders in zip: ['artifact', 'config']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered!
